# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emeka-techDev/ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


Unit of analysis: One row represents one content item for one client on one reporting date.


Tables: I will use fact_content_daily_performance for daily performance data and dim_content for content/search metadata.


Time window: March 2026 will be used as the development window.


Prediction/ranking target: I want to rank content by its likelihood of needing attention/refresh, using future performance change as a proxy label because the warehouse does not contain a direct refresh-success outcome.


Excluded: Future-period performance data will be excluded from the features because it would not be available at the time the ranking decision is made.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {hf_token})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

daily = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"


con.sql("""
SELECT
    month,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY month;
""")


con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1;
""")




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

March 2026 contains 9,841,378 rows in the daily performance table. Also, there are no duplicate combinations of client_hash_id + content_hash_id + report_date in March 2026.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

gsc_impressions

* Available when? Known at the decision moment because it comes from the historical GSC performance data available for the March reporting period.

gsc_clicks

* Available when? Known at the decision moment because it is historical GSC performance for the content.

gsc_avg_position

* Available when? Known at the decision moment because it describes the content's historical search position.

ga4_sessions

* Available when? Known at the decision moment because it is historical GA4 traffic data for the content.

scroll_events

* Available when? Known at the decision moment because it measures historical user interaction with the content.


**Label/Proxy**

*  future performance change (proxy)

Proxy label: future content performance change, since the warehouse does not directly contain a refresh-success outcome.

**Context**
* client_hash_id
* content_hash_id
* report_date
* month

**excluded**
* future-period metrics
* June 2026 data
* content_hash_id
* client_hash_id


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — Grain**

In [ ]:
con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

**Query 2 — Count + date span**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

**Query 3 — GSC availability**

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ gsc_available_rows │
│       int64        │
├────────────────────┤
│            3611061 │
└────────────────────┘

# **Test Leakage**

In [ ]:
df = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
)

SELECT *
FROM march
LIMIT 10;
""")

df.show()

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────────┬──────────────┬───────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ ga4_sessions │ scroll_events │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      double       │    int64     │     int64     │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────────┼──────────────┼───────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │              3.35 │         NULL │          NULL │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │               0.0 │         NULL │          NULL │
│ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │ 2026-03-01  │             125 │        

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(gsc_impressions) AS impressions_present,
    COUNT(gsc_clicks) AS clicks_present,
    COUNT(gsc_avg_position) AS position_present,
    COUNT(ga4_sessions) AS sessions_present,
    COUNT(scroll_events) AS scroll_present

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┬──────────────────┬──────────────────┬────────────────┐
│ total_rows │ impressions_present │ clicks_present │ position_present │ sessions_present │ scroll_present │
│   int64    │        int64        │     int64      │      int64       │      int64       │     int64      │
├────────────┼─────────────────────┼────────────────┼──────────────────┼──────────────────┼────────────────┤
│    9841378 │             9841378 │        9841378 │          3611061 │          6822637 │        6822637 │
└────────────┴─────────────────────┴────────────────┴──────────────────┴──────────────────┴────────────────┘

In [ ]:
con.sql("""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    gsc_data_available,
    ga4_data_available
ORDER BY
    gsc_data_available,
    ga4_data_available;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────────────┬───────────┐
│ gsc_data_available │ ga4_data_available │ row_count │
│      boolean       │      boolean       │   int64   │
├────────────────────┼────────────────────┼───────────┤
│ false              │ false              │   4690323 │
│ false              │ true               │     49619 │
│ false              │ NULL               │   1490375 │
│ true               │ false              │   1718348 │
│ true               │ true               │    364347 │
│ true               │ NULL               │   1528366 │
└────────────────────┴────────────────────┴───────────┘

In [ ]:
con.sql("""
SELECT
    gsc_data_available,
    COUNT(*) AS total_rows,
    COUNT(gsc_avg_position) AS position_present,
    COUNT(*) - COUNT(gsc_avg_position) AS position_missing
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY gsc_data_available
ORDER BY gsc_data_available;
""")

┌────────────────────┬────────────┬──────────────────┬──────────────────┐
│ gsc_data_available │ total_rows │ position_present │ position_missing │
│      boolean       │   int64    │      int64       │      int64       │
├────────────────────┼────────────┼──────────────────┼──────────────────┤
│ false              │    6230317 │                0 │          6230317 │
│ true               │    3611061 │          3611061 │                0 │
└────────────────────┴────────────┴──────────────────┴──────────────────┘

In [ ]:
con.sql("""
SELECT
    ga4_data_available,
    COUNT(*) AS total_rows,
    COUNT(ga4_sessions) AS sessions_present,
    COUNT(*) - COUNT(ga4_sessions) AS sessions_missing,
    COUNT(scroll_events) AS scroll_present,
    COUNT(*) - COUNT(scroll_events) AS scroll_missing
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY ga4_data_available
ORDER BY ga4_data_available;
""")

┌────────────────────┬────────────┬──────────────────┬──────────────────┬────────────────┬────────────────┐
│ ga4_data_available │ total_rows │ sessions_present │ sessions_missing │ scroll_present │ scroll_missing │
│      boolean       │   int64    │      int64       │      int64       │     int64      │     int64      │
├────────────────────┼────────────┼──────────────────┼──────────────────┼────────────────┼────────────────┤
│ false              │    6408671 │          6408671 │                0 │        6408671 │              0 │
│ true               │     413966 │           413966 │                0 │         413966 │              0 │
│ NULL               │    3018741 │                0 │          3018741 │              0 │        3018741 │
└────────────────────┴────────────┴──────────────────┴──────────────────┴────────────────┴────────────────┘

In [ ]:
df = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS NOT NULL;
""")

df.aggregate("COUNT(*) AS row_count").show()

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│   2082695 │
└───────────┘



In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(gsc_impressions) AS impressions_missing,
    COUNT(*) - COUNT(gsc_clicks) AS clicks_missing,
    COUNT(*) - COUNT(gsc_avg_position) AS position_missing,
    COUNT(*) - COUNT(ga4_sessions) AS sessions_missing,
    COUNT(*) - COUNT(scroll_events) AS scroll_missing
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS NOT NULL;
""")

┌────────────┬─────────────────────┬────────────────┬──────────────────┬──────────────────┬────────────────┐
│ total_rows │ impressions_missing │ clicks_missing │ position_missing │ sessions_missing │ scroll_missing │
│   int64    │        int64        │     int64      │      int64       │      int64       │     int64      │
├────────────┼─────────────────────┼────────────────┼──────────────────┼──────────────────┼────────────────┤
│    2082695 │                   0 │              0 │                0 │                0 │              0 │
└────────────┴─────────────────────┴────────────────┴──────────────────┴──────────────────┴────────────────┘

Of all the content items that have usuable march data, how many have data in april

In [6]:
# Of all the content items that have usuable march data, how many have data in april
con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS NOT NULL
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    COUNT(*) AS march_content,
    COUNT(april.content_hash_id) AS content_with_april_data,
    COUNT(*) - COUNT(april.content_hash_id) AS content_without_april_data
FROM march
LEFT JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────────────────┬────────────────────────────┐
│ march_content │ content_with_april_data │ content_without_april_data │
│     int64     │          int64          │           int64            │
├───────────────┼─────────────────────────┼────────────────────────────┤
│        128012 │                  128011 │                          1 │
└───────────────┴─────────────────────────┴────────────────────────────┘

In [ ]:
con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS march_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS NOT NULL
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS april_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    march.client_hash_id,
    march.content_hash_id,
    march.march_sessions,
    april.april_sessions,
    april.april_sessions - march.march_sessions AS sessions_change,
    CASE
        WHEN april.april_sessions > march.march_sessions THEN 1
        ELSE 0
    END AS performance_label
FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
LIMIT 10;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬─────────────────┬───────────────────┐
│     client_hash_id      │     content_hash_id      │ march_sessions │ april_sessions │ sessions_change │ performance_label │
│         varchar         │         varchar          │     int128     │     int128     │     int128      │       int32       │
├─────────────────────────┼──────────────────────────┼────────────────┼────────────────┼─────────────────┼───────────────────┤
│ client_9958f0a7ae1df715 │ content_eb0aeedbcfaf2712 │              4 │              5 │               1 │                 1 │
│ client_9958f0a7ae1df715 │ content_108500096f9bc481 │              7 │              1 │              -6 │                 0 │
│ client_9958f0a7ae1df715 │ content_4cec18f637b4c858 │             10 │              5 │              -5 │                 0 │
│ client_9958f0a7ae1df715 │ content_a600692ebe905459 │              0 │              0 │               0 │     

In [ ]:

# we need to check the label distribution. This is important because we want to know whether 1 and 0 are reasonably represented before using it.
con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS march_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS NOT NULL
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS april_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN april.april_sessions > march.march_sessions THEN 1
        ELSE 0
    END AS performance_label,
    COUNT(*) AS row_count
FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
GROUP BY performance_label
ORDER BY performance_label;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬───────────┐
│ performance_label │ row_count │
│       int32       │   int64   │
├───────────────────┼───────────┤
│                 0 │     77975 │
│                 1 │     50036 │
└───────────────────┴───────────┘

In [ ]:

# Now we combine the March features with the April-based proxy label.
feature_frame = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS NOT NULL

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS april_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    march.client_hash_id,
    march.content_hash_id,

    march.gsc_impressions,
    march.gsc_clicks,
    march.gsc_avg_position,
    march.ga4_sessions,
    march.scroll_events,

    CASE
        WHEN april.april_sessions > march.ga4_sessions THEN 1
        ELSE 0
    END AS performance_label

FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
""")

feature_frame.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬────────────────────┬──────────────┬───────────────┬───────────────────┐
│     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ ga4_sessions │ scroll_events │ performance_label │
│         varchar         │         varchar          │     int128      │   int128   │       double       │    int128    │    int128     │       int32       │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼────────────────────┼──────────────┼───────────────┼───────────────────┤
│ client_9958f0a7ae1df715 │ content_810cf06597918291 │             397 │          1 │ 10.570485279706489 │           42 │            14 │                 0 │
│ client_9958f0a7ae1df715 │ content_1d69c2ed06358f6f │              63 │          0 │ 11.719298245614034 │            6 │             4 │                 0 │
│ client_9958f0a7ae1df715 │ content_7483401e31fc5aa1

In [ ]:

# deliberately create a feature that contains the answer
leaky_frame = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS NOT NULL

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(ga4_sessions) AS april_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    march.*,

    -- DELIBERATE LEAK
    april.april_sessions AS leaky_april_sessions,

    CASE
        WHEN april.april_sessions > march.ga4_sessions THEN 1
        ELSE 0
    END AS performance_label

FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
""")

leaky_frame.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬────────────────────┬──────────────┬───────────────┬──────────────────────┬───────────────────┐
│     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ ga4_sessions │ scroll_events │ leaky_april_sessions │ performance_label │
│         varchar         │         varchar          │     int128      │   int128   │       double       │    int128    │    int128     │        int128        │       int32       │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼────────────────────┼──────────────┼───────────────┼──────────────────────┼───────────────────┤
│ client_9958f0a7ae1df715 │ content_eb0aeedbcfaf2712 │             248 │          0 │ 25.810491753442882 │            4 │             2 │                    5 │                 1 │
│ client_9958f0a7ae1df715 │ content_108500096f9bc481 │             356 │          2 │ 15.280319

In [ ]:


# check how strongly the leaked feature determines the label
con.sql("""
SELECT
    performance_label,
    COUNT(*) AS row_count,
    AVG(leaky_april_sessions) AS avg_april_sessions
FROM leaky_frame
GROUP BY performance_label
ORDER BY performance_label;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬───────────┬────────────────────┐
│ performance_label │ row_count │ avg_april_sessions │
│       int32       │   int64   │       double       │
├───────────────────┼───────────┼────────────────────┤
│                 0 │     77975 │ 3.7532927220262904 │
│                 1 │     50036 │  13.52774002718043 │
└───────────────────┴───────────┴────────────────────┘

In [ ]:
# Run a simple model using the leakage data

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leaky_data = leaky_frame.df()

X = leaky_data[["leaky_april_sessions"]]
y = leaky_data["performance_label"]

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)

predictions = model.predict(X)

accuracy = accuracy_score(y, predictions)

print("Leaky model accuracy:", accuracy)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaky model accuracy: 0.7660044839896571


In [ ]:
# Add ga4_sessions (March) alongside the leaked April value:

X = leaky_data[[
    "ga4_sessions",
    "leaky_april_sessions"
]]

y = leaky_data["performance_label"]

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)

predictions = model.predict(X)

accuracy = accuracy_score(y, predictions)

print("Leaky model accuracy:", accuracy)

Leaky model accuracy: 0.8391935068080086


In [ ]:
# remove the leak and create the honest feature frame
honest_data = feature_frame.df()

X = honest_data[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]]

y = honest_data["performance_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

X shape: (128011, 5)
y shape: (128011,)


In [ ]:

# split into training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (102408, 5)
X_test: (25603, 5)
y_train: (102408,)
y_test: (25603,)


In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Honest model accuracy:", accuracy)

Honest model accuracy: 0.6324649455141975


In [ ]:
print("Final features:")
print(X.columns.tolist())

Final features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation — no direct refresh-success outcome: The warehouse can show content performance and characteristics, but it does not tell us whether refreshing a piece of content actually caused its future performance to improve. Therefore, future performance change is only a proxy for refresh opportunity, not a true measure of refresh success.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.